# Notebook 05 — Tuning des hyperparamètres
## Phase 3 : Optimisation du meilleur modèle

Objectif : optimiser les hyperparamètres de la meilleure combinaison modèle × stratégie identifiée dans le notebook de modélisation, puis valider le résultat sur le jeu de validation.

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
import joblib

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
MODEL_DIR = PROJECT_ROOT / 'models'

train_df = pd.read_csv(DATA_DIR / 'train.csv')
X_train = train_df.drop(columns=['bad_nutrition'])
y_train = train_df['bad_nutrition']
validation_df = pd.read_csv(DATA_DIR / 'validation.csv')
X_val = validation_df.drop(columns=['bad_nutrition'])
y_val = validation_df['bad_nutrition']

model_selection = pd.read_csv(MODEL_DIR / 'model_selection_results.csv')
best_config = model_selection.sort_values(by=['mean_f1', 'std_f1'], ascending=[False, True]).iloc[0]

In [ ]:
best_model_name = best_config['model']
best_strategy = best_config['strategy']

print('Meilleur modèle :', best_model_name)
print('Meilleure stratégie :', best_strategy)
print('F1 moyenne (train CV) :', best_config['mean_f1'])

## 1. Définition du pipeline et des plages d’hyperparamètres

Le tuning se fait sur le modèle retenu, avec une grille justifiée par les recommandations du descriptif de Phase 3.
Chaque plage de recherche est choisie de façon raisonnable pour limiter le surcoût de calcul et rester sur des valeurs pertinentes.

In [ ]:
preprocessor = joblib.load(MODEL_DIR / 'preprocessor.joblib')

def build_pipeline(model, strategy):
    if strategy == 'baseline':
        return Pipeline([('preprocessor', clone(preprocessor)), ('clf', model)])
    sampler = SMOTE(random_state=42) if strategy == 'smote' else RandomUnderSampler(random_state=42)
    return ImbPipeline([('preprocessor', clone(preprocessor)), ('sampler', sampler), ('clf', model)])

if best_model_name == 'LogisticRegression':
    base_model = LogisticRegression(random_state=42, solver='liblinear', max_iter=5000, class_weight='balanced')
    param_distributions = {
        'clf__C': [0.01, 0.1, 1, 10],
        'clf__penalty': ['l1', 'l2']
    }
elif best_model_name == 'DecisionTree':
    base_model = DecisionTreeClassifier(random_state=42, class_weight='balanced')
    param_distributions = {
        'clf__max_depth': [3, 5, 7, 10],
        'clf__min_samples_leaf': [1, 5, 10, 20]
    }
elif best_model_name == 'RandomForest':
    base_model = RandomForestClassifier(random_state=42, class_weight='balanced_subsample', n_estimators=200)
    param_distributions = {
        'clf__n_estimators': [200, 500],
        'clf__max_depth': [10, 20, None],
        'clf__min_samples_leaf': [1, 5, 10],
        'clf__max_features': ['sqrt']
    }
else:
    base_model = MLPClassifier(random_state=42, max_iter=500, early_stopping=True)
    param_distributions = {
        'clf__hidden_layer_sizes': [(128, 64), (128, 64, 32), (64, 64)],
        'clf__activation': ['relu', 'tanh'],
        'clf__alpha': [0.0001, 0.001, 0.01],
        'clf__learning_rate_init': [0.0001, 0.001, 0.01],
        'clf__batch_size': [32, 64, 128]
    }

pipeline = build_pipeline(base_model, best_strategy)
print('Pipeline built for tuning:', pipeline)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring='f1',
    cv=cv,
    random_state=42,
    n_jobs=1,
    verbose=2
)
search.fit(X_train, y_train)

best_model = search.best_estimator_
best_params = search.best_params_
best_score = search.best_score_

print('Best score (CV):', best_score)
print('Best params:')
print(best_params)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_val_pred = best_model.predict(X_val)
report = classification_report(y_val, y_val_pred, target_names=['good', 'bad'])
print('Validation classification report:')
print(report)

print('Validation confusion matrix:')
print(confusion_matrix(y_val, y_val_pred))

In [ ]:
joblib.dump({'pipeline': best_model, 'best_params': best_params, 'best_score': best_score}, MODEL_DIR / 'tuned_model.joblib')
print('Tuned model saved to models/tuned_model.joblib')

## 2. Synthèse du tuning

Le modèle optimisé est sauvegardé dans `models/tuned_model.joblib`.
Utilisez ce pipeline en Phase 3 finale pour l’évaluation sur le jeu de test et l’optimisation du seuil métier.